In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets
import sys
import os
import matplotlib.pyplot as plt
import random
import pickle
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Import custom module
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

%load_ext autoreload
%autoreload 2

from modules.Parameter_Estimation import InteractionSufficientStatistics, ExponentialFamilyModel
from modules.Kl_Computation import compute_exact_kl_divergence
from modules.Experiment_Helpers import (
    set_seed, train_one_epoch, evaluate_and_extract_activations, scale_activations,
    initialize_cdm_layer, optimize_cdm_order_with_tolerance, compute_layer_complexity,
    track_cdm_drift, track_weight_drift
)
from modules.plot_helpers import plot_complexity_dynamics, plot_drift_dynamics

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [ ]:
# define transform and prepare CIFAR10 data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.CIFAR10(root='Analysis/data', train=True, download=False, transform=transform)
test_dataset = datasets.CIFAR10(root='Analysis/data', train=False, download=False, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# check the data type
images, labels = next(iter(train_loader))
print("images:", type(images), " / shape:", images.shape)
print("labels:", type(labels), " / shape:", labels.shape)

images: <class 'torch.Tensor'>  / shape: torch.Size([64, 3, 32, 32])
labels: <class 'torch.Tensor'>  / shape: torch.Size([64])


In [ ]:
# Define a CNN for analysis with 2048 -> 128 -> 32 -> 5 -> 5 -> 10
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super(CIFAR10CNN, self).__init__()
        
        # 1. convolutional layer
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.flatten = nn.Flatten()
        
        # 2. fully-connected layers
        self.fc1 = nn.Linear(2048, 128)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, 5)
        self.fc4 = nn.Linear(5, 5)
        self.fc5 = nn.Linear(5, 10)
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x
        
    def get_activations(self, x):
        conv_out = self.features(x)
        flat = self.flatten(conv_out)
        
        act1 = torch.relu(self.fc1(flat))
        act2 = torch.relu(self.fc2(act1))
        act3 = torch.relu(self.fc3(act2))
        act4 = torch.relu(self.fc4(act3))
        act5 = self.fc5(act4)

        _, predicted = torch.max(act5, 1)

        return {
            'ConvOut': flat.detach().cpu().numpy(), # Convoluted features
            'L1': act1.detach().cpu().numpy(),
            'L2': act2.detach().cpu().numpy(),
            'L3': act3.detach().cpu().numpy(),
            'L4': act4.detach().cpu().numpy(),
            'L5': act5.detach().cpu().numpy(),
            'predicted': predicted.detach().cpu().numpy()
        }

In [ ]:
# ==========================================
# 1. Main Experiment Setup
# ==========================================
max_order = 5
calc_stats = InteractionSufficientStatistics(max_order=max_order)

epochs = 1000  
calc_interval = 100
num_runs = 5
target_layers = ['L3', 'L4']
cdm_lr = 1e-4

# Dynamic mapping: Associates target layer names for analysis with actual variable names in the model
layer_mapping = {
    'L1': 'fc1', 'L2': 'fc2', 'L3': 'fc3', 
    'L4': 'fc4', 'L5': 'fc5', 'L6': 'fc6'
}

final_results = {
    'accuracy': [],
    'L3': [], 'L4': [],
    'drift': {
        'cosine_sim': [],
        'param_velocity': [],
        'param_norm': [],
        'net_weight_cosine_sim': [],
        'net_weight_velocity': [],
        'net_weight_norm': []
    }
}

In [ ]:
save_path = 'Result/pt/CIFAR10/relu/final_results.pt'
start_run = 0

# ==========================================
# 1. backup
# ==========================================
if os.path.exists(save_path):
    print(f"Found existing save file at {save_path}!")
    final_results = torch.load(save_path, map_location=torch.device('cpu'))
    
    start_run = len(final_results['accuracy'])
    
if start_run >= num_runs:
    print("All runs are already completed. Exiting.")

else:
    if start_run > 0:
        print(f"Resuming experiment from Run {start_run + 1}/{num_runs}...")
    else:
        print("No previous save file found. Starting from scratch.")

    # ==========================================
    # 2. Main Execution Loop
    # ==========================================
    for run_idx in range(start_run, num_runs):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n{'='*50}\nStarting Run {run_idx + 1}/{num_runs} (Seed: {current_seed})\n{'='*50}")

        model = CIFAR10CNN().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=0.001)
        
        cdm_models_persist = {layer: {} for layer in target_layers}
        cdm_optimizers_persist = {layer: {} for layer in target_layers}
        is_cdm_initialized = {layer: False for layer in target_layers}
        
        run_complexity_history = {layer: [] for layer in target_layers}
        run_accuracy_history = []
        
        run_drift_metrics = {
            'cosine_sim': {layer: {k: [] for k in range(1, max_order + 1)} for layer in target_layers},
            'param_velocity': {layer: {k: [] for k in range(1, max_order + 1)} for layer in target_layers},
            'param_norm': {layer: {k: [] for k in range(1, max_order + 1)} for layer in target_layers}, 
            'net_weight_cosine_sim': {layer: [] for layer in target_layers},
            'net_weight_velocity': {layer: [] for layer in target_layers},
            'net_weight_norm': {layer: [] for layer in target_layers} 
        }
        
        prev_thetas = {layer: {k: None for k in range(1, max_order + 1)} for layer in target_layers}
        prev_weights = {layer: None for layer in target_layers}

        for epoch in range(epochs):
            
            # ==========================================
            # Phase 1: Learning Phase
            # ==========================================
            avg_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
            
            # ==========================================
            # Phase 2: Obtain Activations & Information Geometry Phase
            # ==========================================
            is_eval_epoch = (epoch == 0) or \
                            (epoch+1 <= 150 and (epoch+1) % 10 == 0) or \
                            (epoch+1 > 150 and (epoch+1) % calc_interval == 0)

            if is_eval_epoch:
                test_acc, layer_activations, predictions = evaluate_and_extract_activations(
                    model, test_loader, device, target_layers
                )
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {test_acc:.2f}%")
                run_accuracy_history.append(test_acc)
                
                for layer_name in target_layers:
                    S_input = scale_activations(layer_activations[layer_name]) 
                    global_pos_mean = initialize_cdm_layer(
                        layer_name, S_input, predictions, epoch+1,
                        cdm_models_persist, cdm_optimizers_persist, is_cdm_initialized, calc_stats, max_order, device, cdm_lr
                    )
                    
                    is_first_cdm_calc = not is_cdm_initialized[layer_name]
                    max_cd_epochs = 500 if is_first_cdm_calc else max(30, int(500 / np.sqrt((epoch + 1) / 10)))
                    
                    for k in range(1, max_order + 1): 
                        # set tolerance to ensure enough CDM epochs
                        optimize_cdm_order_with_tolerance(
                            k, cdm_models_persist[layer_name][k], cdm_models_persist[layer_name][k-1], 
                            cdm_optimizers_persist[layer_name][k], S_input, global_pos_mean[k], 
                            max_cd_epochs, cdm_lr, calc_stats, device, is_init_phase=is_first_cdm_calc,
                            tolerance=5e-4, min_epochs=15
                        )

                    for k in range(1, max_order + 1):
                        theta_norm = torch.norm(cdm_models_persist[layer_name][k].theta, p=2).item()
                        print(f"[{layer_name}] k={k} Theta L2 Norm: {theta_norm:.4f}")

                    c_val = compute_layer_complexity(
                        layer_name, S_input, cdm_models_persist, calc_stats, max_order, device
                    )
                    run_complexity_history[layer_name].append(c_val)

                    with torch.no_grad():
                        track_cdm_drift(layer_name, cdm_models_persist, prev_thetas, run_drift_metrics, max_order)
                        track_weight_drift(model, layer_name, prev_weights, run_drift_metrics, layer_mapping)

        # ----------------------------------------------------
        # Store results in final_results after completing each run
        # ----------------------------------------------------
        final_results['accuracy'].append(run_accuracy_history)
        
        if 'L3' in run_complexity_history and run_complexity_history['L3']:
            final_results['L3'].append(run_complexity_history['L3'])
        if 'L4' in run_complexity_history and run_complexity_history['L4']:
            final_results['L4'].append(run_complexity_history['L4'])
            
        final_results['drift']['cosine_sim'].append(run_drift_metrics['cosine_sim'])
        final_results['drift']['param_velocity'].append(run_drift_metrics['param_velocity'])
        final_results['drift']['param_norm'].append(run_drift_metrics['param_norm'])
        
        final_results['drift']['net_weight_cosine_sim'].append(run_drift_metrics['net_weight_cosine_sim'])
        final_results['drift']['net_weight_velocity'].append(run_drift_metrics['net_weight_velocity'])
        final_results['drift']['net_weight_norm'].append(run_drift_metrics['net_weight_norm'])
        
        # save the result
        os.makedirs('Result/pt/CIFAR10/relu' , exist_ok=True)
        save_path = 'Result/pt/CIFAR10/relu/final_results.pt'
        torch.save(final_results, save_path)
        print(f"Final results successfully saved to: {save_path}")

    print("\nAll runs completed successfully!")

In [ ]:
# load the result if it is not in memory
load_path = 'Result/pt/CIFAR10/relu/final_results.pt'

if 'final_results' not in globals() and 'final_results' not in locals():
    if os.path.exists(load_path):
        final_results = torch.load(load_path, map_location=torch.device('cpu'))
    else:
        print(f"File not found at: {load_path}. Please run the training script first.")

In [ ]:
"""Plot complexity"""
plot_complexity_dynamics(
    final_results=final_results, 
    epochs=epochs, 
    calc_interval=calc_interval, 
    target_layers=target_layers,
    save_path='Result/plot/CIFAR10/relu/complexity_dynamics_plot.png' 
)

In [ ]:
"""Plot dynamics measures"""
plot_drift_dynamics(
    final_results=final_results,
    epochs=epochs,
    calc_interval=calc_interval,
    max_order=max_order,
    target_layers=target_layers,
    save_dir='Result/plot/CIFAR10/relu'
)